# Clothing Classification Model Training

This notebook trains a CNN classifier for garment type detection (tshirt, trousers, other).

**TensorFlow Version Compatibility:**
- This notebook requires **TensorFlow 2.13+** to ensure models are saved in the new format
- Models trained with this notebook will be compatible with FastAPI backend (TF 2.15+)
- Avoids the `batch_shape` compatibility issue from older TensorFlow versions

**Dataset Structure:**
```
clothing.zip
  └── clothing_dataset/
      ├── tshirt/
      │   ├── img1.jpg
      │   └── img2.jpg
      ├── trousers/
      │   ├── img1.jpg
      │   └── img2.jpg
      └── other/ (optional)
          ├── img1.jpg
          └── img2.jpg
```

## 1. Setup and Imports

In [ ]:
import os
import zipfile
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

## 2. Dataset Preparation

In [ ]:
# Dataset paths
zip_path = 'clothing.zip'
extract_path = 'clothing_dataset'

# Extract dataset if needed
if not os.path.exists(extract_path):
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        print("✅ Dataset extracted")
    else:
        raise FileNotFoundError(f"Dataset not found: {zip_path}")
else:
    print(f"✅ Dataset already extracted: {extract_path}")

In [ ]:
# Discover classes
all_dirs = [d for d in os.listdir(extract_path)
            if os.path.isdir(os.path.join(extract_path, d))]

# Verify required classes
required = {'tshirt', 'trousers'}
missing = required - set(all_dirs)
if missing:
    raise ValueError(f"Missing required class folders: {missing}")

# Build class list (stable order)
classes = ['trousers', 'tshirt']  # Keep stable order
if 'other' in all_dirs:
    classes.append('other')
    
num_classes = len(classes)
print(f"\n📊 Classes found: {classes}")
print(f"📊 Number of classes: {num_classes}")

# Count images per class
for cls in classes:
    cls_path = os.path.join(extract_path, cls)
    n_images = len([f for f in os.listdir(cls_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
    print(f"  - {cls}: {n_images} images")

## 3. Data Generators with Augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation generator (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Create generators
train_gen = train_datagen.flow_from_directory(
    extract_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
    classes=classes
)

val_gen = val_datagen.flow_from_directory(
    extract_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
    classes=classes
)

print(f"\n📊 Training samples: {train_gen.samples}")
print(f"📊 Validation samples: {val_gen.samples}")
print(f"\n📊 Class indices: {train_gen.class_indices}")

## 4. Compute Class Weights (Handle Imbalance)

In [ ]:
# Count images per class
class_counts = {}
for cname, cidx in train_gen.class_indices.items():
    cpath = os.path.join(extract_path, cname)
    n = len([f for f in os.listdir(cpath)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
    class_counts[cidx] = n

print("\n📊 Class counts:")
for idx, count in class_counts.items():
    cls_name = [k for k, v in train_gen.class_indices.items() if v == idx][0]
    print(f"  - {cls_name} (index {idx}): {count} images")

# Compute balanced weights
classes_np = np.array(list(class_counts.keys()))
counts_np = np.array(list(class_counts.values()))
cw = compute_class_weight(
    'balanced',
    classes=classes_np,
    y=np.repeat(classes_np, counts_np)
)
class_weight = dict(zip(classes_np, cw))

print("\n⚖️ Class weights (for handling imbalance):")
for idx, weight in class_weight.items():
    cls_name = [k for k, v in train_gen.class_indices.items() if v == idx][0]
    print(f"  - {cls_name} (index {idx}): {weight:.4f}")

## 5. Build CNN Model

**Model Architecture:**
- 4 Convolutional blocks with BatchNorm and Dropout
- Progressive filter increase: 32 → 64 → 128 → 256
- Dense layer with 256 units
- Output layer: softmax (≥3 classes) or sigmoid (2 classes)

In [ ]:
def create_model(input_shape, num_classes):
    """Create CNN model for garment classification."""
    model = Sequential([
        # Block 1
        Conv2D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.2),

        # Block 2
        Conv2D(64, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.3),

        # Block 3
        Conv2D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.3),

        # Block 4
        Conv2D(256, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.4),

        # Dense layers
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),

        # Output layer
        Dense(num_classes, activation=('softmax' if num_classes >= 3 else 'sigmoid'))
    ])
    return model

# Create model
model = create_model(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    num_classes=num_classes
)

# Compile model
if num_classes >= 3:
    loss = 'categorical_crossentropy'
    head_type = 'softmax'
else:
    # Two independent one-vs-rest sigmoids
    loss = 'binary_crossentropy'
    head_type = 'sigmoid_ovr'

model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss=loss,
    metrics=['accuracy']
)

print(f"\n🧠 Model compiled with {head_type} head")
print(f"📊 Loss function: {loss}\n")
model.summary()

## 6. Save Configuration Files

In [ ]:
# Save class labels
with open('class_labels.json', 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print("✅ Saved class_labels.json")
print(json.dumps(train_gen.class_indices, indent=2))

# Save model config
with open('model_config.json', 'w') as f:
    json.dump({'head_type': head_type, 'img_size': IMG_SIZE[0]}, f, indent=2)
print(f"\n✅ Saved model_config.json (head_type={head_type})")

## 7. Setup Callbacks

In [ ]:
callbacks = [
    # Save best model based on validation accuracy
    ModelCheckpoint(
        'best_clothing_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    
    # Stop if no improvement for 15 epochs
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-5,
        verbose=1
    )
]

print("✅ Callbacks configured:")
print("  - ModelCheckpoint: saves best model")
print("  - EarlyStopping: stops if no improvement for 15 epochs")
print("  - ReduceLROnPlateau: reduces LR if validation loss plateaus")

## 8. Train Model

**Training Configuration:**
- Epochs: 50 (with early stopping)
- Batch size: 16
- Learning rate: 5e-4 (with reduction on plateau)
- Class weights: Applied to handle class imbalance

**Note:** Training time depends on dataset size and hardware. With GPU, expect ~30-60 minutes for typical datasets.

In [ ]:
EPOCHS = 50
steps_per_epoch = max(1, train_gen.samples // BATCH_SIZE)
val_steps = max(1, val_gen.samples // BATCH_SIZE)

print(f"\n🚀 Starting training...")
print(f"📊 Epochs: {EPOCHS}")
print(f"📊 Steps per epoch: {steps_per_epoch}")
print(f"📊 Validation steps: {val_steps}\n")

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=val_steps,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

print("\n✅ Training completed!")

## 9. Save Final Model

In [ ]:
# Save final model (may not be the best if early stopping kicked in)
model.save('clothing_model_final.h5')
print("✅ Saved clothing_model_final.h5")

# Verify model files exist
print("\n📁 Model files created:")
for fname in ['best_clothing_model.h5', 'clothing_model_final.h5', 'class_labels.json', 'model_config.json']:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"  ✅ {fname} ({size_mb:.2f} MB)")
    else:
        print(f"  ❌ {fname} (missing)")

## 10. Estimate Rejection Threshold

The rejection threshold (tau) determines the minimum confidence required to accept a prediction.
Lower values = more permissive, higher values = more conservative.

In [ ]:
try:
    # Get one batch from validation set
    val_batch = next(iter(val_gen))
    probs = model.predict(val_batch[0], verbose=0)
    
    # Get max probability for each prediction
    maxp = probs.max(axis=1)
    
    # Set threshold at 10th percentile (conservative)
    tau = max(0.6, float(np.quantile(maxp, 0.10)))
    
    # Save threshold
    with open('rejection_threshold.json', 'w') as f:
        json.dump({'tau': tau}, f, indent=2)
    
    print(f"\n✅ Estimated rejection threshold: tau = {tau:.4f}")
    print(f"   (Saved to rejection_threshold.json)")
    print(f"\n📊 Validation confidence distribution:")
    print(f"   Min:  {maxp.min():.4f}")
    print(f"   10%:  {np.quantile(maxp, 0.10):.4f}")
    print(f"   25%:  {np.quantile(maxp, 0.25):.4f}")
    print(f"   50%:  {np.quantile(maxp, 0.50):.4f}")
    print(f"   75%:  {np.quantile(maxp, 0.75):.4f}")
    print(f"   Max:  {maxp.max():.4f}")
    
except Exception as e:
    print(f"⚠️ Could not estimate tau: {e}")
    print("Using default tau = 0.75")
    with open('rejection_threshold.json', 'w') as f:
        json.dump({'tau': 0.75}, f, indent=2)

## 11. Visualize Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot loss
ax2.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Saved training_history.png")

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"\n📊 Final Metrics:")
print(f"   Training Accuracy:   {final_train_acc:.4f}")
print(f"   Validation Accuracy: {final_val_acc:.4f}")
print(f"   Training Loss:       {final_train_loss:.4f}")
print(f"   Validation Loss:     {final_val_loss:.4f}")

## 12. Test Model Loading (TensorFlow 2.15+ Compatibility)

This verifies that the saved model can be loaded with TensorFlow 2.15+ without the `batch_shape` error.

In [ ]:
print("\n🧪 Testing model loading...\n")

# Test loading best model
try:
    test_model = tf.keras.models.load_model('best_clothing_model.h5', compile=False)
    print("✅ best_clothing_model.h5 loads successfully!")
    print(f"   Input shape:  {test_model.input_shape}")
    print(f"   Output shape: {test_model.output_shape}")
    
    # Test prediction with dummy data
    dummy_input = np.random.rand(1, 224, 224, 3).astype('float32')
    prediction = test_model.predict(dummy_input, verbose=0)
    print(f"\n✅ Test prediction successful!")
    print(f"   Output shape: {prediction.shape}")
    print(f"   Probabilities: {prediction[0]}")
    print(f"   Predicted class: {np.argmax(prediction[0])}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nThis should NOT happen with TensorFlow 2.13+")
    print("If you see this error, you may be using an older TensorFlow version.")

## 13. Summary and Next Steps

### Files Created:
1. **best_clothing_model.h5** - Best model (highest validation accuracy)
2. **clothing_model_final.h5** - Final model after all epochs
3. **class_labels.json** - Class name to index mapping
4. **model_config.json** - Model configuration (head_type, img_size)
5. **rejection_threshold.json** - Confidence threshold for predictions
6. **training_history.png** - Training/validation accuracy and loss plots

### Next Steps:
1. Copy `best_clothing_model.h5` to `models/` directory in FastAPI backend
2. Copy JSON config files to `models/` directory
3. Test classification endpoints in FastAPI
4. Adjust rejection threshold if needed based on production performance

### Copy Files to Backend:
```bash
# From notebook directory
cp best_clothing_model.h5 ../models/
cp class_labels.json ../models/
cp model_config.json ../models/
cp rejection_threshold.json ../models/
```

### Test in FastAPI:
```bash
python test_model_load.py
```

**TensorFlow Version Note:**
- Models trained with this notebook (TF 2.13+) are compatible with your FastAPI backend
- No `batch_shape` compatibility issues
- Ready for deployment to Railway/Heroku